# 🏥 Tech Challenge — Fase 04: Predição de Obesidade

**Pós Tech FIAP | Data Analytics**  
**Aluno:** Gustavo Henrique Lisboa do Nascimento

---

## 📌 Contexto
Cientista de dados de um hospital que precisa desenvolver um modelo de **Machine Learning** para apoiar médicos e médicas na predição de **níveis de obesidade**.

---

## Bibliotecas e configuração

### Objetivo do bloco
Preparar o ambiente: instalar XGBoost, Scikit-learn e importar bibliotecas.

In [ ]:
# Instalações
!pip install -q xgboost
!pip install scikit-learn

In [ ]:
# Imports padrão
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, joblib

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42
print('✅ Bibliotecas carregadas')

---

## Importando e carregando Arquivo


In [ ]:
df_obesity = pd.read_csv('https://drive.google.com/uc?export=download&id=1fgloPUJiXkkccvGfbo5s2M6tXp14L1_Z')
print(f'Shape: {df_obesity.shape}')
df_obesity.head()

---

## Dicionário de Dados + Renomeação para Português

### Dicionário completo

| Coluna original | Nome em português | Tipo | Descrição | Valores |
|---|---|---|---|---|
| `Gender` | `genero` | Nominal binário | Sexo biológico | `Female`, `Male` |
| `Age` | `idade` | Numérico contínuo | Idade em anos | 14 a 61 |
| `Height` | `altura_m` | Numérico contínuo | Altura em metros | 1.45 a 1.98 |
| `Weight` | `peso_kg` | Numérico contínuo | Peso em quilogramas | 39 a 173 |
| `family_history` | `historia_familiar_sobrepeso` | Nominal binário | Algum familiar com excesso de peso? | `yes`, `no` |
| `FAVC` | `come_comida_calorica_freq` | Nominal binário | Consome calóricos com frequência? | `yes`, `no` |
| `FCVC` | `freq_consumo_vegetais` | Ordinal (1-3) | Frequência de vegetais | 1=raro, 2=às vezes, 3=sempre |
| `NCP` | `num_refeicoes_principais` | Ordinal (1-4) | Refeições principais/dia | 1, 2, 3, 4+ |
| `CAEC` | `come_entre_refeicoes` | Ordinal | Lanche entre refeições | `no`, `Sometimes`, `Frequently`, `Always` |
| `SMOKE` | `fumante` | Nominal binário | Fuma? | `yes`, `no` |
| `CH2O` | `consumo_agua_litros` | Ordinal (1-3) | Consumo de água | 1=<1L, 2=1-2L, 3=>2L |
| `SCC` | `monitora_calorias` | Nominal binário | Monitora calorias ingeridas? | `yes`, `no` |
| `FAF` | `freq_atividade_fisica` | Ordinal (0-3) | Atividade física semanal | 0=nenhuma, 1=1-2x, 2=3-4x, 3=5x+ |
| `TUE` | `tempo_uso_dispositivos` | Ordinal (0-2) | Tempo em telas/dispositivos | 0=0-2h, 1=3-5h, 2=>5h |
| `CALC` | `freq_consumo_alcool` | Ordinal | Consumo de álcool | `no`, `Sometimes`, `Frequently`, `Always` |
| `MTRANS` | `meio_transporte` | Nominal categórico | Transporte habitual | `Public_Transportation`, `Walking`, `Automobile`, `Motorbike`, `Bike` |
| `Obesity` | `nivel_obesidade` | **Alvo** (multiclasse, 7 classes) | Classificação do nível corporal | 7 classes (do magro ao obeso III) |


In [ ]:
# Dicionário de renomeação: nome_original → nome_em_português
novos_nomes = {
    'Gender':         'genero',
    'Age':            'idade',
    'Height':         'altura_m',
    'Weight':         'peso_kg',
    'family_history': 'historia_familiar_sobrepeso',
    'FAVC':           'come_comida_calorica_freq',
    'FCVC':           'freq_consumo_vegetais',
    'NCP':            'num_refeicoes_principais',
    'CAEC':           'come_entre_refeicoes',
    'SMOKE':          'fumante',
    'CH2O':           'consumo_agua_litros',
    'SCC':            'monitora_calorias',
    'FAF':            'freq_atividade_fisica',
    'TUE':            'tempo_uso_dispositivos',
    'CALC':           'freq_consumo_alcool',
    'MTRANS':         'meio_transporte',
    'Obesity':        'nivel_obesidade'
}

df_obesity = df_obesity.rename(columns=novos_nomes)
print(f' Colunas renomeadas: {len(novos_nomes)}')
print(f'\nNovas colunas:')
print(df_obesity.columns.tolist())
df_obesity.head()

In [ ]:
# Informações sobre o dataset
print('Tipos:'); print(df_obesity.dtypes)
print(f'\nMissings: {df_obesity.isna().sum().sum()}')
print(f'Duplicatas: {df_obesity.duplicated().sum()}')

In [ ]:
# Descrição sobre as colunas númericas
df_obesity.describe().round(2)

- `idade`: 14 a 61 anos
- `altura_m`: 1.45 a 1.98 m
- `peso_kg`: 39 a 173 kg

In [ ]:
# Validação dos valores únicos
for col in df_obesity.select_dtypes(include='object').columns:
    print(f'{col}: {df_obesity[col].unique().tolist()}')

---

##  Análise Exploratória (EDA)


In [ ]:
# Ordena classes e realiza plot, para validação do balanceamento das classes.
ordem_obesity = [
    'Insufficient_Weight', 'Normal_Weight',
    'Overweight_Level_I', 'Overweight_Level_II',
    'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III'
]

fig, ax = plt.subplots(figsize=(11, 5))
df_obesity['nivel_obesidade'].value_counts().reindex(ordem_obesity).plot(
    kind='bar', color='steelblue', ax=ax
)
ax.set_title('Distribuição das classes de obesidade', weight='bold')
ax.set_ylabel('Quantidade'); ax.set_xlabel('')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

print('\nDistribuição percentual:')
print((df_obesity['nivel_obesidade'].value_counts(normalize=True).reindex(ordem_obesity) * 100).round(2))

### 4.2 IMC — Confirmação do Rótulo

O dataset foi rotulado pela fórmula **IMC = Peso / Altura²**. Isso significa que `peso_kg` e `altura_m` vazam o rótulo.

In [ ]:
# Formúla do IMC
df_obesity['imc'] = df_obesity['peso_kg'] / (df_obesity['altura_m'] ** 2)
df_obesity[['peso_kg', 'altura_m', 'imc', 'nivel_obesidade']].head()

In [ ]:
# Boxplot do IMC para cada uma das 7 classes.
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_obesity, x='nivel_obesidade', y='imc', order=ordem_obesity,
            ax=ax, palette='RdYlGn_r')
ax.set_title('IMC por classe', weight='bold')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
print('\n Obs: Quando uma feature divide as classes em faixas disjuntas, ela define o rótulo. Incluí-la no modelo seria como dar a resposta da prova junto com a pergunta.')

In [ ]:
# Correlação numérica
num_cols = ['idade', 'altura_m', 'peso_kg', 'freq_consumo_vegetais',
            'num_refeicoes_principais', 'consumo_agua_litros',
            'freq_atividade_fisica', 'tempo_uso_dispositivos', 'imc']
corr = df_obesity[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, square=True)
ax.set_title('Matriz de correlação', weight='bold')
plt.tight_layout(); plt.show()

### Hábitos vs Obesidade — Insights para a Equipe Médica

Tradução de dados em decisões clínicas. Médicos não querem ver acurácia — querem saber quais hábitos prevenir.

In [ ]:
# Gráficos Clínicos

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Histórico familiar
pd.crosstab(df_obesity['nivel_obesidade'], df_obesity['historia_familiar_sobrepeso'],
            normalize='index').reindex(ordem_obesity).plot(
    kind='bar', stacked=True, ax=axes[0,0], color=["#3c5be7f9",'#3498db']
)
axes[0,0].set_title('Histórico familiar de excesso de peso', weight='bold')
axes[0,0].set_xlabel(''); axes[0,0].legend(title='Histórico')
axes[0,0].tick_params(axis='x', rotation=30)

# Comida calórica frequente
pd.crosstab(df_obesity['nivel_obesidade'], df_obesity['come_comida_calorica_freq'],
            normalize='index').reindex(ordem_obesity).plot(
    kind='bar', stacked=True, ax=axes[0,1], color=['#3c5be7f9','#3498db']
)
axes[0,1].set_title('Consumo de comida calórica com frequência', weight='bold')
axes[0,1].set_xlabel(''); axes[0,1].legend(title='Calórico?')
axes[0,1].tick_params(axis='x', rotation=30)

# Atividade física
sns.boxplot(data=df_obesity, x='nivel_obesidade', y='freq_atividade_fisica',
            order=ordem_obesity, ax=axes[1,0], palette='Blues')
axes[1,0].set_title('Frequência de atividade física semanal', weight='bold')
axes[1,0].set_xlabel(''); axes[1,0].tick_params(axis='x', rotation=30)

# Meio de transporte
pd.crosstab(df_obesity['nivel_obesidade'], df_obesity['meio_transporte'],
            normalize='index').reindex(ordem_obesity).plot(
    kind='bar', stacked=True, ax=axes[1,1], colormap='tab10'
)
axes[1,1].set_title('Meio de transporte', weight='bold')
axes[1,1].set_xlabel(''); axes[1,1].legend(title='Transporte', bbox_to_anchor=(1.02, 1))
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

### Insights consolidados para a equipe médica

1. **Histórico familiar** — fator não comportamental mais determinante.
2. **Consumo calórico frequente** — prevalente nas classes obesas.
3. **Atividade física baixa** (FAF≤1) — fortemente associada a obesidade.
4. **Transporte ativo** (a pé/bike) é protetor; **automóvel** é fator de risco.

---

## Limpeza e Feature Engineering

In [ ]:
#Remover Duplicados
df_clean = df_obesity.drop_duplicates().reset_index(drop=True).copy()

# Arredondar variáveis ordinais ruidosas
colunas_ordinais = [
    'freq_consumo_vegetais',
    'num_refeicoes_principais',
    'consumo_agua_litros',
    'freq_atividade_fisica',
    'tempo_uso_dispositivos'
]
for col in colunas_ordinais:
    df_clean[col] = df_clean[col].round().astype(int)

# Faixa etária — uso APENAS analítico (gráficos/agrupamentos do dashboard).
# Não entra no modelo: 'idade' contínua é mais informativa, pois preserva
# a granularidade da variável. Discretizar perderia informação que os
# modelos de árvore (RF/XGBoost) já capturam internamente nos splits.
def faixa(a):
    if a < 21: return '<21'
    if a < 30: return '21-29'
    if a < 40: return '30-39'
    if a < 50: return '40-49'
    return '50+'
df_clean['faixa_idade'] = df_clean['idade'].apply(faixa)

print(f'\nShape final: {df_clean.shape}')
df_clean.head()

---

##  Preparação para modelagem

In [ ]:
# Converte `nivel_obesidade` (string) em inteiros 0-6 via `LabelEncoder`

le = LabelEncoder()
df_clean['nivel_obesidade_enc'] = le.fit_transform(df_clean['nivel_obesidade'])
print('Mapeamento do alvo:')
for k, v in dict(zip(le.classes_, le.transform(le.classes_))).items():
    print(f'  {v} → {k}')

###  Definição das features — Modelo A e Modelo B

#### Decisão metodológica central

| Modelo | Features | Por quê? |
|---|---|---|
| **A** (baseline IMC) | Tudo + `peso_kg`, `altura_m`, `imc` | Baseado via IMC |
| **B** (valor clínico) | **Sem** `peso_kg`, `altura_m`, `imc` | Prever a partir de **hábitos** |

Ambos serão treinados, comparados e salvos. Streamlit terá toggle entre A e B.

In [ ]:
# Difine ás duas lista e o mesmo rando_state

features_modelo_A = [
    'genero', 'idade', 'altura_m', 'peso_kg', 'historia_familiar_sobrepeso',
    'come_comida_calorica_freq', 'freq_consumo_vegetais', 'num_refeicoes_principais',
    'come_entre_refeicoes', 'fumante', 'consumo_agua_litros', 'monitora_calorias',
    'freq_atividade_fisica', 'tempo_uso_dispositivos', 'freq_consumo_alcool',
    'meio_transporte', 'imc'
]
features_modelo_B = [
    'genero', 'idade', 'historia_familiar_sobrepeso',
    'come_comida_calorica_freq', 'freq_consumo_vegetais', 'num_refeicoes_principais',
    'come_entre_refeicoes', 'fumante', 'consumo_agua_litros', 'monitora_calorias',
    'freq_atividade_fisica', 'tempo_uso_dispositivos', 'freq_consumo_alcool',
    'meio_transporte'
]

y = df_clean['nivel_obesidade_enc']
X_A = df_clean[features_modelo_A]
X_B = df_clean[features_modelo_B]

# MESMO random_state nos dois → mesmas pessoas em treino/teste
X_A_train, X_A_test, y_train, y_test = train_test_split(
    X_A, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_B_train, X_B_test, _, _ = train_test_split(
    X_B, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
# ColumnTransformer e monta o preprocessador.

def build_preprocessor(X):
    num = X.select_dtypes(include=['int64','float64']).columns.tolist()
    cat = X.select_dtypes(include=['object']).columns.tolist()
    return ColumnTransformer([
        ('num', StandardScaler(), num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)
    ]), num, cat

prep_A, num_A, cat_A = build_preprocessor(X_A_train)
prep_B, num_B, cat_B = build_preprocessor(X_B_train)


---

##  Treinamento

In [ ]:
# Função para validação dos modelos e execução, validado LogReg, RandomForest e XGBoost

def avalia_modelos(X_train, X_test, y_train, y_test, preprocessor, cenario):
    modelos = {
        'LogisticRegression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=6,
                                  random_state=RANDOM_STATE, eval_metric='mlogloss', n_jobs=-1)
    }
    resultados, pipelines = [], {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for nome, clf in modelos.items():
        pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
        cv_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        resultados.append({
            'Cenario': cenario, 'Modelo': nome, 'CV_acc': cv_acc,
            'Test_acc': accuracy_score(y_test, y_pred),
            'Test_F1_macro': f1_score(y_test, y_pred, average='macro')
        })
        pipelines[nome] = pipe
        print(f'  {nome:20s} | CV: {cv_acc:.4f} | Test: {accuracy_score(y_test,y_pred):.4f} | F1: {f1_score(y_test,y_pred,average="macro"):.4f}')
    return pd.DataFrame(resultados), pipelines

print('  MODELO A — COM peso_kg/altura_m/imc (baseline trivial)')
print('-' * 72)
res_A, pipes_A = avalia_modelos(X_A_train, X_A_test, y_train, y_test, prep_A, 'A')

print('\n  MODELO B — SEM peso_kg/altura_m/imc (valor clínico)')
print('-' * 72)
res_B, pipes_B = avalia_modelos(X_B_train, X_B_test, y_train, y_test, prep_B, 'B')

In [ ]:
# Resumo das comprações

resumo = pd.concat([res_A, res_B], ignore_index=True)
resumo_pivot = resumo.pivot(index='Modelo', columns='Cenario',
                             values=['CV_acc','Test_acc','Test_F1_macro']).round(4)
print('Resumo comparativo:')
display(resumo_pivot)

In [ ]:
# Gráficos dos modelos

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, metric in enumerate(['Test_acc','Test_F1_macro']):
    pivot = resumo.pivot(index='Modelo', columns='Cenario', values=metric)
    pivot.plot(kind='bar', ax=axes[i], color=['#3498db','#e74c3c'])
    axes[i].set_title(f'{metric} — A vs B', weight='bold')
    axes[i].axhline(0.75, color='green', linestyle='--', alpha=0.6, label='Critério (75%)')
    axes[i].legend(title='Cenário'); axes[i].set_ylabel(metric); axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=20); axes[i].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

**Modelo A acerta ~98% porque "decora" o IMC** — sem mérito algum.


**Modelo B acerta ~78% a partir só de hábitos** — esta é a métrica clinicamente útil.

In [ ]:
#Identificar o melhor modelo

champ_A_name = res_A.sort_values('CV_acc', ascending=False).iloc[0]['Modelo']
champ_B_name = res_B.sort_values('CV_acc', ascending=False).iloc[0]['Modelo']
print(f' Campeão A: {champ_A_name}')
print(f' Campeão B: {champ_B_name}')
champion_A = pipes_A[champ_A_name]

In [ ]:
# GridSearch adaptado ao algoritmo campeão
if champ_B_name == 'RandomForest':
    param_grid = {
        'clf__n_estimators': [200, 400, 600],
        'clf__max_depth': [None, 10, 20],
        'clf__min_samples_split': [2, 5],
        'clf__min_samples_leaf': [1, 2]
    }
    base_clf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
elif champ_B_name == 'XGBoost':
    param_grid = {
        'clf__n_estimators': [200, 400],
        'clf__max_depth': [4, 6, 8],
        'clf__learning_rate': [0.05, 0.1],
        'clf__subsample': [0.8, 1.0]
    }
    base_clf = XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss', n_jobs=-1)
else:
    param_grid = {'clf__C': [0.1, 1.0, 10.0], 'clf__penalty': ['l2'], 'clf__solver': ['lbfgs', 'saga']}
    base_clf = LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)

pipe_B_tune = Pipeline([('prep', prep_B), ('clf', base_clf)])
grid = GridSearchCV(pipe_B_tune, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_B_train, y_train)
print(f'\n✅ Melhores hiperparâmetros: {grid.best_params_}')
print(f'✅ Melhor CV accuracy: {grid.best_score_:.4f}')
champion_B = grid.best_estimator_

In [ ]:
y_pred_A = champion_A.predict(X_A_test)
y_pred_B = champion_B.predict(X_B_test)
acc_A, f1_A = accuracy_score(y_test, y_pred_A), f1_score(y_test, y_pred_A, average='macro')
acc_B, f1_B = accuracy_score(y_test, y_pred_B), f1_score(y_test, y_pred_B, average='macro')
print(f'🅰️  Modelo A ({champ_A_name}): acc={acc_A:.4f} | F1={f1_A:.4f}')
print(f'🅱️  Modelo B ({champ_B_name} tunado): acc={acc_B:.4f} | F1={f1_B:.4f}')
print(f'\n{"✅ PASSOU" if acc_B>0.75 else "❌ FALHOU"} no critério >75%')

In [ ]:
# Matriz de confusão do Modelo B
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_B,
    display_labels=le.classes_,
    cmap='Blues',
    xticks_rotation=45,
    ax=ax
)
ax.set_title(f'Matriz de Confusão — Modelo B ({champ_B_name})', weight='bold')
plt.tight_layout(); plt.show()

---

## Preparação para o Streamlit


In [ ]:
# Pipeline completa dos modelos, tradução, hiperparâmetros e dicionário.
os.makedirs('artifacts', exist_ok=True)
joblib.dump(champion_A, 'artifacts/modelo_A_com_imc.pkl')
joblib.dump(champion_B, 'artifacts/modelo_B_sem_imc.pkl')
joblib.dump(le, 'artifacts/label_encoder.pkl')

# Validação: confirma que os pipelines salvos contêm o preprocessador
print('Etapas do champion_B:', [step[0] for step in champion_B.steps])

# Dicionário de tradução das colunas (para uso no app Streamlit)
dicionario_colunas = {
    'genero': {'orig': 'Gender', 'desc': 'Sexo biológico'},
    'idade': {'orig': 'Age', 'desc': 'Idade em anos'},
    'altura_m': {'orig': 'Height', 'desc': 'Altura em metros'},
    'peso_kg': {'orig': 'Weight', 'desc': 'Peso em quilogramas'},
    'historia_familiar_sobrepeso': {'orig': 'family_history', 'desc': 'Histórico familiar de excesso de peso'},
    'come_comida_calorica_freq': {'orig': 'FAVC', 'desc': 'Consome alimentos calóricos com frequência?'},
    'freq_consumo_vegetais': {'orig': 'FCVC', 'desc': 'Frequência de consumo de vegetais (1-3)'},
    'num_refeicoes_principais': {'orig': 'NCP', 'desc': 'Número de refeições principais por dia (1-4)'},
    'come_entre_refeicoes': {'orig': 'CAEC', 'desc': 'Lanche entre refeições'},
    'fumante': {'orig': 'SMOKE', 'desc': 'Fuma?'},
    'consumo_agua_litros': {'orig': 'CH2O', 'desc': 'Consumo diário de água (1=<1L, 2=1-2L, 3=>2L)'},
    'monitora_calorias': {'orig': 'SCC', 'desc': 'Monitora calorias ingeridas?'},
    'freq_atividade_fisica': {'orig': 'FAF', 'desc': 'Frequência semanal de atividade física (0-3)'},
    'tempo_uso_dispositivos': {'orig': 'TUE', 'desc': 'Tempo em dispositivos (0=0-2h, 1=3-5h, 2=>5h)'},
    'freq_consumo_alcool': {'orig': 'CALC', 'desc': 'Frequência de consumo de álcool'},
    'meio_transporte': {'orig': 'MTRANS', 'desc': 'Meio de transporte habitual'},
    'nivel_obesidade': {'orig': 'Obesity', 'desc': 'Nível de obesidade (alvo)'}
}

metadata = {
    'modelo_A': {
        'nome': champ_A_name, 'features': features_modelo_A,
        'test_accuracy': float(acc_A), 'test_f1_macro': float(f1_A),
        'descricao': 'Com peso/altura/IMC'
    },
    'modelo_B': {
        'nome': champ_B_name, 'features': features_modelo_B,
        'test_accuracy': float(acc_B), 'test_f1_macro': float(f1_B),
        'best_params': {k: str(v) for k, v in grid.best_params_.items()},
        'descricao': 'Sem peso/altura — valor clínico real'
    },
    'classes': le.classes_.tolist(),
    'dicionario_colunas': dicionario_colunas
}
with open('artifacts/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(' Artefatos salvos em artifacts/')
print('   • modelo_A_com_imc.pkl')
print('   • modelo_B_sem_imc.pkl')
print('   • label_encoder.pkl')
print('   • metadata.json')